In [4]:

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score


In [22]:
df = pd.read_csv("train.csv")


In [6]:
X = df[['Sex', 'Embarked', 'Age', 'Fare']].copy()

y = df['Survived']

print("Input Features:")
print(X.head())

print("\nTarget:")
print(y.head())

print("\nInput Shape:", X.shape)
print("Target Shape:", y.shape)

Input Features:
      Sex Embarked   Age     Fare
0    male        S  22.0   7.2500
1  female        C  38.0  71.2833
2  female        S  26.0   7.9250
3  female        S  35.0  53.1000
4    male        S  35.0   8.0500

Target:
0    0
1    1
2    1
3    1
4    0
Name: Survived, dtype: int64

Input Shape: (891, 4)
Target Shape: (891,)


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (712, 4)
Testing: (179, 4)


In [8]:
print("Missing values in each feature:")
print(X_train.isnull().sum())

Missing values in each feature:
Sex           0
Embarked      2
Age         137
Fare          0
dtype: int64


In [9]:
age_imputer = SimpleImputer(strategy='median')
embarked_imputer = SimpleImputer(strategy='most_frequent')

X_train['Age'] = age_imputer.fit_transform(X_train[['Age']]).ravel()
X_test['Age'] = age_imputer.transform(X_test[['Age']]).ravel()

X_train['Embarked'] = embarked_imputer.fit_transform(X_train[['Embarked']]).ravel()
X_test['Embarked'] = embarked_imputer.transform(X_test[['Embarked']]).ravel()

print("Missing values after imputation:")
print(X_train.isnull().sum())

Missing values after imputation:
Sex         0
Embarked    0
Age         0
Fare        0
dtype: int64


In [10]:
sex_encoder = LabelEncoder()

X_train['Sex'] = sex_encoder.fit_transform(X_train['Sex'])
X_test['Sex'] = sex_encoder.transform(X_test['Sex'])

print("Label Encoding:")
print(dict(zip(sex_encoder.classes_, sex_encoder.transform(sex_encoder.classes_))))

print("\nUpdated Training Data:")
print(X_train.head())

Label Encoding:
{'female': np.int64(0), 'male': np.int64(1)}

Updated Training Data:
     Sex Embarked   Age      Fare
692    1        S  28.5   56.4958
481    1        S  28.5    0.0000
527    1        S  28.5  221.7792
855    0        S  18.0    9.3500
801    0        S  31.0   26.2500


In [11]:
embarked_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

train_encoded = embarked_encoder.fit_transform(X_train[['Embarked']])
test_encoded = embarked_encoder.transform(X_test[['Embarked']])

embarked_columns = embarked_encoder.get_feature_names_out(['Embarked'])

X_train[embarked_columns] = train_encoded
X_test[embarked_columns] = test_encoded

X_train.drop('Embarked', axis=1, inplace=True)
X_test.drop('Embarked', axis=1, inplace=True)

print("Encoded Columns:")
print(embarked_columns)

print("\nUpdated Training Data:")
print(X_train.head())

Encoded Columns:
['Embarked_C' 'Embarked_Q' 'Embarked_S']

Updated Training Data:
     Sex   Age      Fare  Embarked_C  Embarked_Q  Embarked_S
692    1  28.5   56.4958         0.0         0.0         1.0
481    1  28.5    0.0000         0.0         0.0         1.0
527    1  28.5  221.7792         0.0         0.0         1.0
855    0  18.0    9.3500         0.0         0.0         1.0
801    0  31.0   26.2500         0.0         0.0         1.0


In [12]:
scaler = StandardScaler()

X_train[['Age', 'Fare']] = scaler.fit_transform(
    X_train[['Age', 'Fare']]
)

X_test[['Age', 'Fare']] = scaler.transform(
    X_test[['Age', 'Fare']]
)

print("Updated Training Data:")
print(X_train.head())

print("\nMean of scaled features:")
print(X_train[['Age', 'Fare']].mean())

print("\nStandard deviation:")
print(X_train[['Age', 'Fare']].std())

Updated Training Data:
     Sex       Age      Fare  Embarked_C  Embarked_Q  Embarked_S
692    1 -0.081135  0.513812         0.0         0.0         1.0
481    1 -0.081135 -0.662563         0.0         0.0         1.0
527    1 -0.081135  3.955399         0.0         0.0         1.0
855    0 -0.887827 -0.467874         0.0         0.0         1.0
801    0  0.110934 -0.115977         0.0         0.0         1.0

Mean of scaled features:
Age     7.734138e-17
Fare   -1.746418e-17
dtype: float64

Standard deviation:
Age     1.000703
Fare    1.000703
dtype: float64


In [13]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Logistic Regression Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100)

Logistic Regression Accuracy: 0.776536312849162
Accuracy Percentage: 77.6536312849162


In [14]:
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

print(df[['SibSp', 'Parch', 'FamilySize']].head(10))

   SibSp  Parch  FamilySize
0      1      0           2
1      1      0           2
2      0      0           1
3      1      0           2
4      0      0           1
5      0      0           1
6      0      0           1
7      3      1           5
8      0      2           3
9      1      0           2


In [15]:
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

print(df[['FamilySize', 'IsAlone']].head(10))

   FamilySize  IsAlone
0           2        0
1           2        0
2           1        1
3           2        0
4           1        1
5           1        1
6           1        1
7           5        0
8           3        0
9           2        0


In [16]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.')

print(df[['Name', 'Title']].head(10))

                                                Name   Title
0                            Braund, Mr. Owen Harris      Mr
1  Cumings, Mrs. John Bradley (Florence Briggs Th...     Mrs
2                             Heikkinen, Miss. Laina    Miss
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)     Mrs
4                           Allen, Mr. William Henry      Mr
5                                   Moran, Mr. James      Mr
6                            McCarthy, Mr. Timothy J      Mr
7                     Palsson, Master. Gosta Leonard  Master
8  Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)     Mrs
9                Nasser, Mrs. Nicholas (Adele Achem)     Mrs


In [17]:
X_train['FamilySize'] = df.loc[X_train.index, 'FamilySize']
X_test['FamilySize'] = df.loc[X_test.index, 'FamilySize']

X_train['IsAlone'] = df.loc[X_train.index, 'IsAlone']
X_test['IsAlone'] = df.loc[X_test.index, 'IsAlone']

print("Updated Training Data:")
print(X_train.head())

print("\nTraining Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

Updated Training Data:
     Sex       Age      Fare  Embarked_C  Embarked_Q  Embarked_S  FamilySize  IsAlone
692    1 -0.081135  0.513812         0.0         0.0         1.0           1        1
481    1 -0.081135 -0.662563         0.0         0.0         1.0           1        1
527    1 -0.081135  3.955399         0.0         0.0         1.0           1        1
855    0 -0.887827 -0.467874         0.0         0.0         1.0           2        0
801    0  0.110934 -0.115977         0.0         0.0         1.0           3        0

Training Data Shape: (712, 8)
Testing Data Shape: (179, 8)


In [18]:
title_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

train_title_encoded = title_encoder.fit_transform(
    df.loc[X_train.index, ['Title']]
)

test_title_encoded = title_encoder.transform(
    df.loc[X_test.index, ['Title']]
)

title_columns = title_encoder.get_feature_names_out(['Title'])

X_train[title_columns] = train_title_encoded
X_test[title_columns] = test_title_encoded

print("Encoded Title Columns:")
print(title_columns)

print("\nUpdated Training Data:")
print(X_train.head())

Encoded Title Columns:
['Title_Col' 'Title_Don' 'Title_Dr' 'Title_Jonkheer' 'Title_Lady'
 'Title_Major' 'Title_Master' 'Title_Miss' 'Title_Mlle' 'Title_Mr'
 'Title_Mrs' 'Title_Ms' 'Title_Rev' 'Title_Sir']

Updated Training Data:
     Sex       Age      Fare  Embarked_C  Embarked_Q  Embarked_S  FamilySize  IsAlone  Title_Col  Title_Don  Title_Dr  Title_Jonkheer  Title_Lady  Title_Major  Title_Master  Title_Miss  Title_Mlle  Title_Mr  Title_Mrs  Title_Ms  Title_Rev  Title_Sir
692    1 -0.081135  0.513812         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
481    1 -0.081135 -0.662563         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
527    1 -0.081135  3

In [19]:
title_encoder = OneHotEncoder(
    handle_unknown='ignore',
    sparse_output=False
)

train_title_encoded = title_encoder.fit_transform(
    df.loc[X_train.index, ['Title']]
)

test_title_encoded = title_encoder.transform(
    df.loc[X_test.index, ['Title']]
)

title_columns = title_encoder.get_feature_names_out(['Title'])

X_train[title_columns] = train_title_encoded
X_test[title_columns] = test_title_encoded

print("Encoded Title Columns:")
print(title_columns)

print("\nUpdated Training Data:")
print(X_train.head())

Encoded Title Columns:
['Title_Col' 'Title_Don' 'Title_Dr' 'Title_Jonkheer' 'Title_Lady'
 'Title_Major' 'Title_Master' 'Title_Miss' 'Title_Mlle' 'Title_Mr'
 'Title_Mrs' 'Title_Ms' 'Title_Rev' 'Title_Sir']

Updated Training Data:
     Sex       Age      Fare  Embarked_C  Embarked_Q  Embarked_S  FamilySize  IsAlone  Title_Col  Title_Don  Title_Dr  Title_Jonkheer  Title_Lady  Title_Major  Title_Master  Title_Miss  Title_Mlle  Title_Mr  Title_Mrs  Title_Ms  Title_Rev  Title_Sir
692    1 -0.081135  0.513812         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
481    1 -0.081135 -0.662563         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
527    1 -0.081135  3

In [20]:
from sklearn.preprocessing import StandardScaler

In [21]:
scaler_new = StandardScaler()

X_train[['Age', 'Fare']] = scaler_new.fit_transform(
    X_train[['Age', 'Fare']]
)

X_test[['Age', 'Fare']] = scaler_new.transform(
    X_test[['Age', 'Fare']]
)

print("Updated Training Data:")
print(X_train.head())

print("\nMean of scaled features:")
print(X_train[['Age', 'Fare']].mean())

Updated Training Data:
     Sex       Age      Fare  Embarked_C  Embarked_Q  Embarked_S  FamilySize  IsAlone  Title_Col  Title_Don  Title_Dr  Title_Jonkheer  Title_Lady  Title_Major  Title_Master  Title_Miss  Title_Mlle  Title_Mr  Title_Mrs  Title_Ms  Title_Rev  Title_Sir
692    1 -0.081135  0.513812         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
481    1 -0.081135 -0.662563         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0        0.0        0.0
527    1 -0.081135  3.955399         0.0         0.0         1.0           1        1        0.0        0.0       0.0             0.0         0.0          0.0           0.0         0.0         0.0       1.0        0.0       0.0

In [23]:
model_new = LogisticRegression(max_iter=1000)

model_new.fit(X_train, y_train)

y_pred_new = model_new.predict(X_test)

accuracy_new = accuracy_score(y_test, y_pred_new)

print("New Logistic Regression Accuracy:", accuracy_new)
print("Accuracy Percentage:", accuracy_new * 100)

New Logistic Regression Accuracy: 0.8324022346368715
Accuracy Percentage: 83.24022346368714


In [24]:
baseline_accuracy = accuracy

print("Original Model Accuracy:",
      baseline_accuracy * 100)

print("Feature Engineered Model Accuracy:",
      accuracy_new * 100)

print("\nAccuracy Difference:",
      (accuracy_new - baseline_accuracy) * 100)

Original Model Accuracy: 77.6536312849162
Feature Engineered Model Accuracy: 83.24022346368714

Accuracy Difference: 5.586592178770944
